# Demo — orquestador multi-agente de Rumbo Sur Viajes

Una consulta real recorre el grafo: el supervisor delega en el investigador, el analista y el planificador, revisa el resultado con la rúbrica y termina. Corrido una vez con Gemini (free tier) y Pinecone.

In [1]:
import logging
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.rate_limiters import InMemoryRateLimiter

from agent.graph import build_llm
from cloud_rag.connection import connect_pinecone
from cloud_rag.ingest import load_chunk_metas
from cloud_rag.rag_system import RAGSystem
from multiagent.graph import build_travel_graph

logging.basicConfig(level=logging.WARNING)
load_dotenv()
llm = build_llm()
llm.rate_limiter = InMemoryRateLimiter(requests_per_second=10 / 60, max_bucket_size=1)
index, embeddings, namespace = connect_pinecone()
rag = RAGSystem(index=index, embeddings=embeddings, namespace=namespace, metas=load_chunk_metas(Path("data")))
app = build_travel_graph(llm, rag).compile()

## Grafo

In [2]:
print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	researcher(researcher)
	analyst(analyst)
	planner(planner)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	analyst --> supervisor;
	planner --> supervisor;
	researcher --> supervisor;
	supervisor -. &nbsp;FINISH&nbsp; .-> __end__;
	supervisor -.-> analyst;
	supervisor -.-> planner;
	supervisor -.-> researcher;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Consulta

In [3]:
question = (
    "Quiero el paquete Cusco y Machu Picchu para 2 personas con upgrade de hotel. "
    "¿Cuánto sale, cuánto es la seña y qué hacemos cada día?"
)
result = await app.ainvoke({"messages": [HumanMessage(question)]})

## Delegación paso a paso

Cada mensaje lleva el nombre de quien lo escribió. Las líneas `[supervisor]` son las decisiones del router (structured output con `SupervisorDecision`).

In [4]:
for message in result["messages"][1:]:
    print(f"[{message.name}] {message.text}")
    print()

[supervisor] -> researcher: Como las notas están pendientes, se necesita al investigador para buscar la información inicial en la documentación.

[researcher] - **Paquete Cusco y Machu Picchu:** 7 noches. [paquetes_y_destinos.md]
- **Precio por persona:** 1.950 dólares. [paquetes_y_destinos.md]
- **Upgrade de hotel (Mejora de categoría de hotel):** 35 dólares por noche y por persona. [paquetes_y_destinos.md]
- **Condiciones de pago:** Seña del 30% del valor del paquete para confirmar la reserva. El saldo debe pagarse como máximo 20 días antes de la salida. [paquetes_y_destinos.md]
- **Reglas, plazos y recomendaciones del paquete:**
  - Las entradas a Machu Picchu tienen cupo diario, por lo que la reserva debe confirmarse al menos 45 días antes de la salida. [paquetes_y_destinos.md]
  - Por la altura de Cusco, se recomienda no planificar actividades exigentes el primer día. [paquetes_y_destinos.md]
- **Qué incluye el paquete:** Hotel tres estrellas con desayuno, tren a Aguas Calientes, 

## Estado final

Cada agente escribió solo su campo: `research_notes`, `cost_estimate` (validado con Pydantic por la tool) e `itinerary`.

In [5]:
print("cost_estimate:", repr(result["cost_estimate"]))
print("pasos del supervisor:", result["step_count"])
print()
print(result["itinerary"])

cost_estimate: CostEstimate(people=2, total=4390.0, deposit=1317.0)
pasos del supervisor: 4

Día 1: Llegada a Cusco, traslado del aeropuerto al hotel y descanso.
Día 2: Excursión al Valle Sagrado.
Día 3: Tren a Aguas Calientes.
Día 4: Entrada a Machu Picchu con guía.
Día 5: Día libre.
Día 6: Día libre.
Día 7: Día libre.
Día 8: Traslado del hotel al aeropuerto y regreso.
